In [ ]:
import os
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
PARQUET_PATH = DATA_DIR / "set_price_window.parquet"
REFRESH = False  # set True to re-query the DB

DB_CONFIG = dict(
    host="localhost",
    port=5433,
    dbname="automana",
    user="automana_admin",
    password=os.environ.get("AUTOMANA_DB_PASSWORD", ""),
)

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

In [ ]:
SQL = """
SELECT
    s.set_code,
    s.set_name,
    s.released_at,
    r.rarity_name,
    cv.card_version_id,
    ppd.price_date,
    EXTRACT(DAY FROM (ppd.price_date - s.released_at::date))::int AS days_since_release,
    ppd.list_low_cents
FROM pricing.print_price_daily ppd
JOIN card_catalog.card_version cv       ON cv.card_version_id = ppd.card_version_id
JOIN card_catalog.rarities_ref r        ON r.rarity_id = cv.rarity_id
JOIN card_catalog.sets s                ON s.set_id = cv.set_id
JOIN card_catalog.set_type_list_ref st  ON st.set_type_id = s.set_type_id
WHERE st.set_type = 'expansion'
  AND s.released_at BETWEEN '2021-01-01' AND '2025-01-01'
  AND ppd.price_date >= s.released_at
  AND ppd.price_date <= s.released_at + INTERVAL '150 days'
  AND ppd.finish_id = 1
  AND ppd.list_low_cents IS NOT NULL
  AND ppd.list_low_cents > 0
  AND r.rarity_name IN ('common', 'uncommon', 'rare', 'mythic')
"""

if REFRESH or not PARQUET_PATH.exists():
    print("Querying DB…")
    with get_conn() as conn:
        raw = pd.read_sql(SQL, conn, parse_dates=["released_at", "price_date"])
    raw.to_parquet(PARQUET_PATH, index=False)
    print(f"Saved {len(raw):,} rows → {PARQUET_PATH}")
else:
    raw = pd.read_parquet(PARQUET_PATH)
    print(f"Loaded {len(raw):,} rows from parquet")
    assert len(raw) > 0, "Parquet loaded empty — delete it and re-run with REFRESH=True"

print(raw.head(3))

In [ ]:
print("Sets in data:", raw["set_code"].nunique())
print("\nRarity breakdown:\n", raw["rarity_name"].value_counts())
print("\nDays range:", raw["days_since_release"].min(), "→", raw["days_since_release"].max())
print("Null list_low_cents:", raw["list_low_cents"].isna().sum())

sets_summary = (
    raw.groupby(["set_code", "set_name", "released_at"])
    .agg(cards=("card_version_id", "nunique"), rows=("list_low_cents", "count"))
    .sort_values("released_at")
)
display(sets_summary)

In [ ]:
# Median list_low_cents across all cards in the group for each day
daily = (
    raw.groupby(["set_code", "set_name", "released_at", "rarity_name", "days_since_release"])
    .agg(
        median_cents=("list_low_cents", "median"),
        card_count=("card_version_id", "nunique"),
    )
    .reset_index()
)

# Drop groups with fewer than 5 distinct cards (thin markets / promos slipping through)
daily = daily[daily["card_count"] >= 5].copy()

print(f"Rows after aggregation: {len(daily):,}")
print(daily.head(10))

In [ ]:
def day0_price(group):
    anchor = group[group["days_since_release"] <= 7].nsmallest(1, "days_since_release")
    if anchor.empty:
        return pd.Series(np.nan, index=group.index)
    base = anchor["median_cents"].values[0]
    if base == 0:
        return pd.Series(np.nan, index=group.index)
    return (group["median_cents"] / base) * 100

daily["price_index"] = (
    daily.groupby(["set_code", "rarity_name"], group_keys=False)
    .apply(day0_price)
)

# Drop series that couldn't be anchored
daily = daily.dropna(subset=["price_index"])
print(f"Rows after normalisation: {len(daily):,}")
print(daily[daily["days_since_release"] == 0][["set_code", "rarity_name", "price_index"]].head(8))

In [ ]:
daily = daily.sort_values(["set_code", "rarity_name", "days_since_release"])

def _smooth_roc(g):
    """Reindex to calendar-day range so rolling window = real 7 days."""
    g_idx = g.set_index("days_since_release").reindex(range(0, 151))
    g_idx["smoothed"] = g_idx["price_index"].rolling(window=7, min_periods=3, center=True).median()
    g_idx["roc_14d"] = g_idx["smoothed"].pct_change(periods=14) * 100
    return g_idx.reset_index().rename(columns={"days_since_release": "days_since_release"})

daily = (
    daily.groupby(["set_code", "set_name", "released_at", "rarity_name"], group_keys=False)
    .apply(_smooth_roc)
    # Drop gap-fill rows (days that had no real observations)
    .dropna(subset=["median_cents"])
    .reset_index(drop=True)
)

# Spot-check: one set, one rarity
sample = daily[(daily["set_code"] == "dmu") & (daily["rarity_name"] == "rare")]
print(sample[["days_since_release", "price_index", "smoothed", "roc_14d"]].head(15))

In [ ]:
print(daily[daily["set_code"] == "dmu"][
    ["days_since_release", "rarity_name", "smoothed", "roc_14d"]
].dropna().head(20))

In [ ]:
THRESHOLD = -5.0    # ROC above this = "stabilized"
MIN_DECLINE_DAYS = 7

def find_inflection(group):
    g = group.dropna(subset=["roc_14d"]).sort_values("days_since_release")
    if g.empty:
        return np.nan

    streak_start_day = None
    entered_decline = False
    for _, row in g.iterrows():
        if row["roc_14d"] < THRESHOLD:
            if streak_start_day is None:
                streak_start_day = row["days_since_release"]
            if (row["days_since_release"] - streak_start_day) >= MIN_DECLINE_DAYS:
                entered_decline = True
        else:
            if entered_decline:
                return row["days_since_release"]
            streak_start_day = None

    # Fallback: day of minimum smoothed index
    return g.loc[g["smoothed"].idxmin(), "days_since_release"]

inflections = (
    daily.groupby(["set_code", "set_name", "released_at", "rarity_name"])
    .apply(find_inflection, include_groups=False)
    .reset_index(name="inflection_day")
    .dropna(subset=["inflection_day"])
)
inflections["inflection_day"] = inflections["inflection_day"].astype(int)

print(inflections.groupby("rarity_name")["inflection_day"].describe().round(1))

In [ ]:
RARITY_ORDER = ["mythic", "rare", "uncommon", "common"]
RARITY_COLORS = {"mythic": "#e07b39", "rare": "#c5a800", "uncommon": "#8fb5c2", "common": "#9e9e9e"}

fig, ax = plt.subplots(figsize=(12, 6))

for rarity in RARITY_ORDER:
    sub = daily[daily["rarity_name"] == rarity]
    agg = sub.groupby("days_since_release")["smoothed"].agg(
        median="median",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
    )
    agg = agg[agg.index <= 150]
    color = RARITY_COLORS[rarity]
    ax.plot(agg.index, agg["median"], label=rarity.capitalize(), color=color, linewidth=2)
    ax.fill_between(agg.index, agg["p25"], agg["p75"], color=color, alpha=0.12)

ax.axhline(100, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_xlabel("Days since set release")
ax.set_ylabel("Price index (day-0 = 100)")
ax.set_title("Normalized price trajectory after set release\n(expansion sets 2021\u20132024, nonfoil, shaded = IQR)")
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))
plt.tight_layout()
plt.savefig(DATA_DIR / "fig1_avg_price_curve.png", dpi=150)
plt.show()

In [ ]:
EXAMPLE_SET = "blb"   # Bloomburrow — change to any set_code in the data

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

for rarity in RARITY_ORDER:
    sub = daily[(daily["set_code"] == EXAMPLE_SET) & (daily["rarity_name"] == rarity)].sort_values("days_since_release")
    if sub.empty:
        continue
    color = RARITY_COLORS[rarity]
    ax1.plot(sub["days_since_release"], sub["smoothed"], color=color, linewidth=2, label=rarity.capitalize())
    ax2.plot(sub["days_since_release"], sub["roc_14d"], color=color, linewidth=1, linestyle=":", alpha=0.6)

    infl = inflections[(inflections["set_code"] == EXAMPLE_SET) & (inflections["rarity_name"] == rarity)]
    if not infl.empty:
        day = infl["inflection_day"].values[0]
        ax1.axvline(day, color=color, linestyle="--", linewidth=1, alpha=0.7)

ax1.axhline(100, color="black", linestyle="--", linewidth=0.8, alpha=0.4)
ax2.axhline(-5, color="grey", linestyle=":", linewidth=0.8, alpha=0.6)
ax1.set_xlabel("Days since release")
ax1.set_ylabel("Price index (day-0 = 100)")
ax2.set_ylabel("14-day ROC (%)")
ax1.legend(loc="upper right")

set_rows = daily[daily["set_code"] == EXAMPLE_SET]
if set_rows.empty:
    raise ValueError(f"EXAMPLE_SET '{EXAMPLE_SET}' not found in daily — check card_count filter or parquet")
set_name = set_rows["set_name"].iloc[0]
ax1.set_title(f"{set_name} — price index + rate of change\n(dashed verticals = detected inflection day)")
plt.tight_layout()
plt.savefig(DATA_DIR / f"fig2_roc_{EXAMPLE_SET}.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)
axes = axes.flatten()

for i, rarity in enumerate(RARITY_ORDER):
    ax = axes[i]
    sub = inflections[inflections["rarity_name"] == rarity]["inflection_day"]
    if sub.empty:
        continue
    ax.hist(sub, bins=range(0, 155, 7), color=RARITY_COLORS[rarity], edgecolor="white", alpha=0.85)
    median_day = int(sub.median())
    ax.axvline(median_day, color="black", linestyle="--", linewidth=1.2)
    ax.set_title(f"{rarity.capitalize()}  (n={len(sub)}, median={median_day}d)")
    ax.set_xlabel("Inflection day")
    ax.set_ylabel("Number of sets")
    ax.set_xlim(0, 150)

fig.suptitle("Distribution of price bottom (inflection day) by rarity\nExpansion sets 2021\u20132024", fontsize=13)
plt.tight_layout()
plt.savefig(DATA_DIR / "fig3_inflection_distribution.png", dpi=150)
plt.show()

In [ ]:
summary = (
    inflections.groupby("rarity_name")["inflection_day"]
    .agg(
        sets="count",
        median="median",
        p25=lambda x: x.quantile(0.25),
        p75=lambda x: x.quantile(0.75),
        min="min",
        max="max",
    )
    .loc[RARITY_ORDER]
    .round(1)
)
summary.index.name = "Rarity"
summary.columns = ["Sets", "Median day", "25th pct", "75th pct", "Min", "Max"]
display(summary)

print("\nConclusion:")
for rarity in RARITY_ORDER:
    row = summary.loc[rarity]
    print(f"  {rarity.capitalize():10s} → prices bottom around day {int(row['Median day'])} "
          f"(IQR {int(row['25th pct'])}–{int(row['75th pct'])})")